In [3]:
from pathlib import Path

# The notebook is inside the notebooks/ folder.
# parent gives us the project root directory.
PROJECT_ROOT = Path.cwd().parent

RAW_FLIGHTS_DIR = PROJECT_ROOT / "data" / "raw" / "flights_kaggle"
CSV_PATH = RAW_FLIGHTS_DIR / "flights_sample_3m.csv"
DICTIONARY_PATH = RAW_FLIGHTS_DIR / "dictionary.html"

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV path: {CSV_PATH}")
print(f"Dictionary path: {DICTIONARY_PATH}")

assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"
assert DICTIONARY_PATH.exists(), f"Dictionary not found: {DICTIONARY_PATH}"

print("Both raw flight files were found successfully.")

Project root: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence
CSV path: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\raw\flights_kaggle\flights_sample_3m.csv
Dictionary path: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\raw\flights_kaggle\dictionary.html
Both raw flight files were found successfully.


In [4]:
import hashlib
from datetime import datetime, timezone

def compute_sha256(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    """
    Calculate the SHA-256 hash of a file without loading the whole file into memory.

    Parameters
    ----------
    file_path : Path
        Path to the file to hash.
    chunk_size : int
        Number of bytes read at a time.

    Returns
    -------
    str
        SHA-256 hexadecimal digest.
    """
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


csv_file_size_bytes = CSV_PATH.stat().st_size
dictionary_file_size_bytes = DICTIONARY_PATH.stat().st_size

csv_sha256 = compute_sha256(CSV_PATH)
dictionary_sha256 = compute_sha256(DICTIONARY_PATH)

audit_started_at_utc = datetime.now(timezone.utc).isoformat()

print("Raw flight source file information")
print("-" * 60)
print(f"Audit timestamp (UTC): {audit_started_at_utc}")
print(f"CSV filename: {CSV_PATH.name}")
print(f"CSV size: {csv_file_size_bytes:,} bytes ({csv_file_size_bytes / 1024**2:.2f} MB)")
print(f"CSV SHA-256: {csv_sha256}")
print()
print(f"Dictionary filename: {DICTIONARY_PATH.name}")
print(
    f"Dictionary size: {dictionary_file_size_bytes:,} bytes "
    f"({dictionary_file_size_bytes / 1024**2:.2f} MB)"
)
print(f"Dictionary SHA-256: {dictionary_sha256}")

Raw flight source file information
------------------------------------------------------------
Audit timestamp (UTC): 2026-09-20T21:41:41.436286+00:00
CSV filename: flights_sample_3m.csv
CSV size: 614,140,650 bytes (585.69 MB)
CSV SHA-256: 66c338ff6c99db352ceee1af0d3e41329cf466c1c203abee52aec1dc56fe7dc4

Dictionary filename: dictionary.html
Dictionary size: 5,011 bytes (0.00 MB)
Dictionary SHA-256: b278560237ee39604563cfde566afd8dbdd242b2ee0e9f29716c72565410db3a


In [5]:
import csv
import pandas as pd

# Read only a small sample first.
# This confirms the delimiter, headers, inferred data types, and example values
# without loading the 585 MB source CSV into memory.
SAMPLE_ROW_COUNT = 10

sample_flights = pd.read_csv(
    CSV_PATH,
    nrows=SAMPLE_ROW_COUNT,
    low_memory=False
)

print("Raw CSV schema preview")
print("-" * 60)
print(f"Number of columns: {sample_flights.shape[1]}")
print(f"Preview rows loaded: {sample_flights.shape[0]}")

print("\nColumn names:")
for column_number, column_name in enumerate(sample_flights.columns, start=1):
    print(f"{column_number:>3}. {column_name}")

print("\nInferred data types:")
print(sample_flights.dtypes.to_string())

print("\nFirst 10 raw records:")
display(sample_flights)

# Count data rows using Python's csv module.
# This is memory-safe because it reads the file one record at a time.
with CSV_PATH.open(
    mode="r",
    encoding="utf-8",
    newline=""
) as csv_file:
    raw_row_count = sum(1 for _ in csv.reader(csv_file)) - 1

print("\nRaw source row count")
print("-" * 60)
print(f"Data rows, excluding the header: {raw_row_count:,}")

Raw CSV schema preview
------------------------------------------------------------
Number of columns: 32
Preview rows loaded: 10

Column names:
  1. FL_DATE
  2. AIRLINE
  3. AIRLINE_DOT
  4. AIRLINE_CODE
  5. DOT_CODE
  6. FL_NUMBER
  7. ORIGIN
  8. ORIGIN_CITY
  9. DEST
 10. DEST_CITY
 11. CRS_DEP_TIME
 12. DEP_TIME
 13. DEP_DELAY
 14. TAXI_OUT
 15. WHEELS_OFF
 16. WHEELS_ON
 17. TAXI_IN
 18. CRS_ARR_TIME
 19. ARR_TIME
 20. ARR_DELAY
 21. CANCELLED
 22. CANCELLATION_CODE
 23. DIVERTED
 24. CRS_ELAPSED_TIME
 25. ELAPSED_TIME
 26. AIR_TIME
 27. DISTANCE
 28. DELAY_DUE_CARRIER
 29. DELAY_DUE_WEATHER
 30. DELAY_DUE_NAS
 31. DELAY_DUE_SECURITY
 32. DELAY_DUE_LATE_AIRCRAFT

Inferred data types:
FL_DATE                     object
AIRLINE                     object
AIRLINE_DOT                 object
AIRLINE_CODE                object
DOT_CODE                     int64
FL_NUMBER                    int64
ORIGIN                      object
ORIGIN_CITY                 object
DEST               

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN
5,2019-07-31,Southwest Airlines Co.,Southwest Airlines Co.: WN,WN,19393,665,DAL,"Dallas, TX",OKC,"Oklahoma City, OK",...,0.0,60.0,54.0,36.0,181.0,141.0,0.0,0.0,0.0,0.0
6,2023-06-11,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,2134,DCA,"Washington, DC",BOS,"Boston, MA",...,0.0,109.0,89.0,58.0,399.0,NaN,NaN,NaN,NaN,NaN
7,2019-07-08,Republic Airline,Republic Airline: YX,YX,20452,4464,HSV,"Huntsville, AL",DCA,"Washington, DC",...,0.0,122.0,151.0,88.0,613.0,0.0,0.0,23.0,0.0,0.0
8,2023-02-12,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,590,IAH,"Houston, TX",LAX,"Los Angeles, CA",...,0.0,227.0,219.0,200.0,1379.0,NaN,NaN,NaN,NaN,NaN
9,2020-08-22,Alaska Airlines Inc.,Alaska Airlines Inc.: AS,AS,19930,223,SEA,"Seattle, WA",FAI,"Fairbanks, AK",...,0.0,210.0,220.0,198.0,1533.0,NaN,NaN,NaN,NaN,NaN



Raw source row count
------------------------------------------------------------
Data rows, excluding the header: 3,000,000


In [6]:
from collections import Counter

# Configuration
CHUNK_SIZE = 100_000

# Fields that must be audited across the entire raw file.
AUDIT_COLUMNS = [
    "FL_DATE",
    "AIRLINE",
    "AIRLINE_DOT",
    "AIRLINE_CODE",
    "DOT_CODE",
    "FL_NUMBER",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "ARR_TIME",
    "ARR_DELAY",
    "CANCELLED",
    "CANCELLATION_CODE",
    "DIVERTED",
]

# Validate that every requested audit field is present in the raw source schema.
missing_audit_columns = sorted(set(AUDIT_COLUMNS) - set(sample_flights.columns))

if missing_audit_columns:
    raise ValueError(
        "The following requested audit columns are not present in the raw CSV: "
        f"{missing_audit_columns}"
    )

# Accumulators for full-file results.
total_rows = 0
null_counts = Counter()
minimum_flight_date = None
maximum_flight_date = None

cancelled_flight_count = 0
diverted_flight_count = 0
completed_flight_count = 0

unique_airlines = set()
unique_origins = set()
unique_destinations = set()

# Read only the audit fields, one chunk at a time.
for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        usecols=AUDIT_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    total_rows += len(chunk)

    # Count null values in each audited field.
    for column_name in AUDIT_COLUMNS:
        null_counts[column_name] += int(chunk[column_name].isna().sum())

    # Parse the flight-date field safely and update source coverage.
    flight_dates = pd.to_datetime(
        chunk["FL_DATE"],
        errors="coerce",
    )

    valid_flight_dates = flight_dates.dropna()

    if not valid_flight_dates.empty:
        chunk_minimum_date = valid_flight_dates.min()
        chunk_maximum_date = valid_flight_dates.max()

        if (
            minimum_flight_date is None
            or chunk_minimum_date < minimum_flight_date
        ):
            minimum_flight_date = chunk_minimum_date

        if (
            maximum_flight_date is None
            or chunk_maximum_date > maximum_flight_date
        ):
            maximum_flight_date = chunk_maximum_date

    # Convert indicator fields safely because raw values may be floats or missing.
    cancelled_indicator = chunk["CANCELLED"].fillna(0).eq(1)
    diverted_indicator = chunk["DIVERTED"].fillna(0).eq(1)

    cancelled_flight_count += int(cancelled_indicator.sum())
    diverted_flight_count += int(diverted_indicator.sum())

    # A provisional completed-flight cohort for this audit:
    # not cancelled, not diverted, and final arrival delay is available.
    completed_indicator = (
        ~cancelled_indicator
        & ~diverted_indicator
        & chunk["ARR_DELAY"].notna()
    )
    completed_flight_count += int(completed_indicator.sum())

    # Capture coverage of airline and airport identifiers.
    unique_airlines.update(chunk["AIRLINE"].dropna().astype(str).unique())
    unique_origins.update(chunk["ORIGIN"].dropna().astype(str).unique())
    unique_destinations.update(chunk["DEST"].dropna().astype(str).unique())

    # Progress message every 10 chunks, i.e. roughly every 1 million rows.
    if chunk_number % 10 == 0:
        print(
            f"Processed {total_rows:,} rows "
            f"across {chunk_number:,} chunks."
        )

# Create a reproducible full-file missingness table.
missingness_audit = pd.DataFrame(
    {
        "column": AUDIT_COLUMNS,
        "null_count": [null_counts[column] for column in AUDIT_COLUMNS],
        "null_fraction": [
            null_counts[column] / total_rows
            for column in AUDIT_COLUMNS
        ],
    }
).sort_values(
    by="null_fraction",
    ascending=False,
).reset_index(drop=True)

# Calculate audit-level rates.
cancelled_rate = cancelled_flight_count / total_rows
diverted_rate = diverted_flight_count / total_rows
completed_rate = completed_flight_count / total_rows

print("\nFull raw-flight source audit")
print("-" * 60)
print(f"Total data rows: {total_rows:,}")
print(
    "Actual flight-date coverage: "
    f"{minimum_flight_date.date()} to {maximum_flight_date.date()}"
)
print(f"Unique airline names: {len(unique_airlines):,}")
print(f"Unique origin airports: {len(unique_origins):,}")
print(f"Unique destination airports: {len(unique_destinations):,}")
print(f"Cancelled flights: {cancelled_flight_count:,} ({cancelled_rate:.2%})")
print(f"Diverted flights: {diverted_flight_count:,} ({diverted_rate:.2%})")
print(
    "Provisional completed flights "
    "(not cancelled, not diverted, ARR_DELAY present): "
    f"{completed_flight_count:,} ({completed_rate:.2%})"
)

print("\nField-level missingness")
display(
    missingness_audit.style.format(
        {
            "null_count": "{:,.0f}",
            "null_fraction": "{:.2%}",
        }
    )
)

# Save a small, Git-trackable audit output for Phase 1 documentation.
AUDIT_OUTPUT_PATH = PROJECT_ROOT / "data" / "external" / "flight_missingness_audit.csv"
AUDIT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

missingness_audit.to_csv(AUDIT_OUTPUT_PATH, index=False)

print(f"\nSaved missingness audit: {AUDIT_OUTPUT_PATH}")

Processed 1,000,000 rows across 10 chunks.
Processed 2,000,000 rows across 20 chunks.
Processed 3,000,000 rows across 30 chunks.

Full raw-flight source audit
------------------------------------------------------------
Total data rows: 3,000,000
Actual flight-date coverage: 2019-01-01 to 2023-08-31
Unique airline names: 18
Unique origin airports: 380
Unique destination airports: 380
Cancelled flights: 79,140 (2.64%)
Diverted flights: 7,056 (0.24%)
Provisional completed flights (not cancelled, not diverted, ARR_DELAY present): 2,913,802 (97.13%)

Field-level missingness


,column,null_count,null_fraction
0,CANCELLATION_CODE,"2,920,860",97.36%
1,ARR_DELAY,"86,198",2.87%
2,ARR_TIME,"79,942",2.66%
3,DEP_TIME,"77,615",2.59%
4,FL_DATE,0,0.00%
5,AIRLINE,0,0.00%
6,AIRLINE_DOT,0,0.00%
7,ORIGIN,0,0.00%
8,FL_NUMBER,0,0.00%
9,DOT_CODE,0,0.00%



Saved missingness audit: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\flight_missingness_audit.csv


In [7]:
# Read the supplied HTML data dictionary.
# pandas.read_html returns a list because a single HTML page can contain
# multiple tables. This file contains the flight-field dictionary table.
dictionary_tables = pd.read_html(DICTIONARY_PATH)

if len(dictionary_tables) != 1:
    raise ValueError(
        "Expected exactly one table in dictionary.html, "
        f"but found {len(dictionary_tables)} tables."
    )

flight_dictionary = dictionary_tables[0].copy()

# Standardise dictionary column names for reliable downstream use.
flight_dictionary.columns = (
    flight_dictionary.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Confirm that the supplied dictionary has the fields required for the audit.
required_dictionary_columns = {
    "updated_header",
    "source_header",
    "data_type",
    "description",
}

missing_dictionary_columns = (
    required_dictionary_columns
    - set(flight_dictionary.columns)
)

if missing_dictionary_columns:
    raise ValueError(
        "The dictionary is missing required columns: "
        f"{sorted(missing_dictionary_columns)}"
    )

# Add an explicit flag showing whether each documented field exists
# in the raw CSV that was actually audited.
raw_csv_columns = set(sample_flights.columns)

flight_dictionary["present_in_raw_csv"] = (
    flight_dictionary["updated_header"]
    .astype(str)
    .str.strip()
    .isin(raw_csv_columns)
)

# Create the project's initial canonical field mapping.
# Canonical names are descriptive, snake_case names used later in
# standardised tables. No raw data values are changed in this step.
canonical_name_mapping = {
    "FL_DATE": "flight_date_local",
    "AIRLINE": "airline_name",
    "AIRLINE_DOT": "airline_dot_name",
    "AIRLINE_CODE": "reporting_airline_code",
    "DOT_CODE": "reporting_airline_dot_id",
    "FL_NUMBER": "flight_number",
    "ORIGIN": "origin_airport",
    "ORIGIN_CITY": "origin_city",
    "DEST": "destination_airport",
    "DEST_CITY": "destination_city",
    "CRS_DEP_TIME": "scheduled_departure_time_local_hhmm",
    "DEP_TIME": "actual_departure_time_local_hhmm",
    "DEP_DELAY": "departure_delay_minutes",
    "TAXI_OUT": "taxi_out_minutes",
    "WHEELS_OFF": "wheels_off_time_local_hhmm",
    "WHEELS_ON": "wheels_on_time_local_hhmm",
    "TAXI_IN": "taxi_in_minutes",
    "CRS_ARR_TIME": "scheduled_arrival_time_local_hhmm",
    "ARR_TIME": "actual_arrival_time_local_hhmm",
    "ARR_DELAY": "arrival_delay_minutes",
    "CANCELLED": "cancelled_indicator",
    "CANCELLATION_CODE": "cancellation_reason_code",
    "DIVERTED": "diverted_indicator",
    "CRS_ELAPSED_TIME": "scheduled_elapsed_minutes",
    "ELAPSED_TIME": "actual_elapsed_minutes",
    "AIR_TIME": "air_time_minutes",
    "DISTANCE": "distance_miles",
    "DELAY_DUE_CARRIER": "delay_due_carrier_minutes",
    "DELAY_DUE_WEATHER": "delay_due_weather_minutes",
    "DELAY_DUE_NAS": "delay_due_national_air_system_minutes",
    "DELAY_DUE_SECURITY": "delay_due_security_minutes",
    "DELAY_DUE_LATE_AIRCRAFT": "delay_due_late_aircraft_minutes",
}

flight_dictionary["canonical_field_name"] = (
    flight_dictionary["updated_header"]
    .map(canonical_name_mapping)
)

# Determine whether a tail/aircraft identifier exists under common possible names.
possible_tail_identifier_fields = {
    "TAIL_NUM",
    "TAIL_NUMBER",
    "TAILNUMBER",
    "AIRCRAFT_ID",
    "AIRCRAFT_IDENTIFIER",
}

tail_identifier_columns_found = sorted(
    raw_csv_columns.intersection(possible_tail_identifier_fields)
)

has_tail_identifier = len(tail_identifier_columns_found) > 0

# Identify the raw columns that exist but are undocumented in dictionary.html,
# plus fields described in the dictionary but absent from the raw CSV.
dictionary_raw_headers = set(flight_dictionary["updated_header"].astype(str))

undocumented_raw_columns = sorted(
    raw_csv_columns - dictionary_raw_headers
)

dictionary_fields_missing_from_raw_csv = sorted(
    dictionary_raw_headers - raw_csv_columns
)

# Save the field mapping as a small Git-trackable documentation artefact.
FIELD_MAPPING_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "flight_field_mapping.csv"
)

flight_dictionary.to_csv(
    FIELD_MAPPING_OUTPUT_PATH,
    index=False,
)

print("Data dictionary validation")
print("-" * 60)
print(f"Dictionary fields: {len(flight_dictionary):,}")
print(f"Raw CSV fields: {len(raw_csv_columns):,}")
print(
    "Dictionary fields present in raw CSV: "
    f"{flight_dictionary['present_in_raw_csv'].sum():,}"
)
print(f"Undocumented raw CSV fields: {len(undocumented_raw_columns):,}")
print(
    "Dictionary fields absent from raw CSV: "
    f"{len(dictionary_fields_missing_from_raw_csv):,}"
)

print("\nAircraft-identifier audit")
print("-" * 60)
print(f"Tail/aircraft identifier available: {has_tail_identifier}")
print(
    "Matching tail/aircraft fields found: "
    f"{tail_identifier_columns_found if tail_identifier_columns_found else 'None'}"
)

print("\nScope decision for this audited CSV")
if has_tail_identifier:
    print(
        "A tail/aircraft identifier is available. "
        "Rotation eligibility can be assessed after timestamp normalisation."
    )
else:
    print(
        "No tail/aircraft identifier is available in flights_sample_3m.csv. "
        "Aircraft-rotation, aircraft-level propagation, and rotation-based "
        "recovery analysis are not supported by this source file."
    )

print("\nMapping file saved:")
print(FIELD_MAPPING_OUTPUT_PATH)

print("\nInitial field mapping preview:")
display(
    flight_dictionary[
        [
            "updated_header",
            "canonical_field_name",
            "data_type",
            "description",
            "present_in_raw_csv",
        ]
    ]
)

Data dictionary validation
------------------------------------------------------------
Dictionary fields: 30
Raw CSV fields: 32
Dictionary fields present in raw CSV: 30
Undocumented raw CSV fields: 2
Dictionary fields absent from raw CSV: 0

Aircraft-identifier audit
------------------------------------------------------------
Tail/aircraft identifier available: False
Matching tail/aircraft fields found: None

Scope decision for this audited CSV
No tail/aircraft identifier is available in flights_sample_3m.csv. Aircraft-rotation, aircraft-level propagation, and rotation-based recovery analysis are not supported by this source file.

Mapping file saved:
c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\flight_field_mapping.csv

Initial field mapping preview:


,updated_header,canonical_field_name,data_type,description,present_in_raw_csv
0,FL_DATE,flight_date_local,object,Flight Date (yyyymmdd),True
1,AIRLINE_CODE,reporting_airline_code,object,Unique Carrier Code. When the same code has be...,True
2,DOT_CODE,reporting_airline_dot_id,int64,An identification number assigned by US DOT to...,True
3,FL_NUMBER,flight_number,int64,Flight Number,True
4,ORIGIN,origin_airport,object,Origin Airport,True
5,ORIGIN_CITY,origin_city,object,"Origin Airport, City Name",True
6,DEST,destination_airport,object,Destination Airport,True
7,DEST_CITY,destination_city,object,"Destination Airport, City Name",True
8,CRS_DEP_TIME,scheduled_departure_time_local_hhmm,int64,CRS Departure Time (local time: hhmm),True
9,DEP_TIME,actual_departure_time_local_hhmm,float64,Actual Departure Time (local time: hhmm),True


In [8]:
# Audit all local clock-time fields across the full raw CSV.
# The source dictionary defines these values as local times in HHMM format.
# This step does not modify any data; it only measures raw-data quality.

TIME_COLUMNS = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "WHEELS_OFF",
    "WHEELS_ON",
    "CRS_ARR_TIME",
    "ARR_TIME",
]

# Confirm that all expected time fields exist before auditing.
missing_time_columns = sorted(set(TIME_COLUMNS) - raw_csv_columns)

if missing_time_columns:
    raise ValueError(
        "The following required time fields are missing from the raw CSV: "
        f"{missing_time_columns}"
    )

# Define accepted raw HHMM values:
# - 0000 through 2359 are valid same-day local clock times.
# - 2400 is accepted as midnight at the start of the following local day.
# - Hours must be 00–23 and minutes must be 00–59.
def classify_hhmm_value(value) -> str:
    """
    Classify a raw flight clock-time value.

    Returns one of:
    - missing
    - non_integer
    - valid_hhmm
    - valid_2400
    - invalid_hhmm
    """
    if pd.isna(value):
        return "missing"

    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return "non_integer"

    if not numeric_value.is_integer():
        return "non_integer"

    hhmm = int(numeric_value)

    if hhmm == 2400:
        return "valid_2400"

    if hhmm < 0 or hhmm > 2359:
        return "invalid_hhmm"

    hour = hhmm // 100
    minute = hhmm % 100

    if hour <= 23 and minute <= 59:
        return "valid_hhmm"

    return "invalid_hhmm"


# Accumulate raw time-quality counts and examples without retaining the full file.
time_quality_counts = {
    column_name: {
        "missing": 0,
        "non_integer": 0,
        "valid_hhmm": 0,
        "valid_2400": 0,
        "invalid_hhmm": 0,
    }
    for column_name in TIME_COLUMNS
}

invalid_time_examples = {
    column_name: []
    for column_name in TIME_COLUMNS
}

MAX_EXAMPLES_PER_COLUMN = 10

for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        usecols=TIME_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    for column_name in TIME_COLUMNS:
        classifications = chunk[column_name].apply(classify_hhmm_value)

        classification_counts = classifications.value_counts()

        for category_name, count in classification_counts.items():
            time_quality_counts[column_name][category_name] += int(count)

        # Save a few invalid raw examples for diagnosis.
        invalid_mask = classifications.isin(
            ["non_integer", "invalid_hhmm"]
        )

        available_example_slots = (
            MAX_EXAMPLES_PER_COLUMN
            - len(invalid_time_examples[column_name])
        )

        if available_example_slots > 0 and invalid_mask.any():
            examples = (
                chunk.loc[invalid_mask, column_name]
                .head(available_example_slots)
                .tolist()
            )
            invalid_time_examples[column_name].extend(examples)

    if chunk_number % 10 == 0:
        print(
            f"Audited raw clock times for {chunk_number * CHUNK_SIZE:,} "
            "rows."
        )

# Build a clean table for documentation and later ingestion rules.
time_quality_rows = []

for column_name in TIME_COLUMNS:
    counts = time_quality_counts[column_name]
    total_field_rows = sum(counts.values())

    for category_name, count in counts.items():
        time_quality_rows.append(
            {
                "raw_time_column": column_name,
                "category": category_name,
                "count": count,
                "fraction": count / total_field_rows,
                "example_invalid_values": (
                    invalid_time_examples[column_name]
                    if category_name in {"non_integer", "invalid_hhmm"}
                    else None
                ),
            }
        )

time_quality_audit = pd.DataFrame(time_quality_rows)

TIME_AUDIT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "flight_time_quality_audit.csv"
)

time_quality_audit.to_csv(
    TIME_AUDIT_OUTPUT_PATH,
    index=False,
)

print("\nRaw HHMM time-field quality audit")
print("-" * 60)

for column_name in TIME_COLUMNS:
    counts = time_quality_counts[column_name]
    total_field_rows = sum(counts.values())

    valid_count = (
        counts["valid_hhmm"]
        + counts["valid_2400"]
    )

    invalid_count = (
        counts["non_integer"]
        + counts["invalid_hhmm"]
    )

    print(f"\n{column_name}")
    print(f"  Valid HHMM values: {valid_count:,} ({valid_count / total_field_rows:.2%})")
    print(f"  Valid 2400 values: {counts['valid_2400']:,} ({counts['valid_2400'] / total_field_rows:.2%})")
    print(f"  Missing values: {counts['missing']:,} ({counts['missing'] / total_field_rows:.2%})")
    print(f"  Invalid values: {invalid_count:,} ({invalid_count / total_field_rows:.2%})")

    if invalid_time_examples[column_name]:
        print(
            "  Invalid examples: "
            f"{invalid_time_examples[column_name]}"
        )

print(f"\nSaved time-quality audit: {TIME_AUDIT_OUTPUT_PATH}")

Audited raw clock times for 1,000,000 rows.
Audited raw clock times for 2,000,000 rows.
Audited raw clock times for 3,000,000 rows.

Raw HHMM time-field quality audit
------------------------------------------------------------

CRS_DEP_TIME
  Valid HHMM values: 3,000,000 (100.00%)
  Valid 2400 values: 0 (0.00%)
  Missing values: 0 (0.00%)
  Invalid values: 0 (0.00%)

DEP_TIME
  Valid HHMM values: 2,922,385 (97.41%)
  Valid 2400 values: 242 (0.01%)
  Missing values: 77,615 (2.59%)
  Invalid values: 0 (0.00%)

WHEELS_OFF
  Valid HHMM values: 2,921,194 (97.37%)
  Valid 2400 values: 401 (0.01%)
  Missing values: 78,806 (2.63%)
  Invalid values: 0 (0.00%)

WHEELS_ON
  Valid HHMM values: 2,920,056 (97.34%)
  Valid 2400 values: 1,109 (0.04%)
  Missing values: 79,944 (2.66%)
  Invalid values: 0 (0.00%)

CRS_ARR_TIME
  Valid HHMM values: 3,000,000 (100.00%)
  Valid 2400 values: 14 (0.00%)
  Missing values: 0 (0.00%)
  Invalid values: 0 (0.00%)

ARR_TIME
  Valid HHMM values: 2,920,058 (97.34%)


In [9]:
# Audit scheduled departure/arrival time consistency.
#
# The raw source stores scheduled departure and arrival as local HHMM values,
# while CRS_ELAPSED_TIME gives scheduled duration in minutes.
#
# We do NOT yet construct timezone-aware timestamps here. This audit identifies:
# - flights whose scheduled arrival clock time is earlier than departure,
#   which usually indicates a local-date rollover or timezone difference;
# - whether adding one calendar day makes the raw clock difference broadly
#   consistent with the reported scheduled elapsed time;
# - records requiring careful treatment during later UTC normalisation.

SCHEDULE_AUDIT_COLUMNS = [
    "FL_DATE",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "ORIGIN",
    "DEST",
]

missing_schedule_columns = sorted(
    set(SCHEDULE_AUDIT_COLUMNS) - raw_csv_columns
)

if missing_schedule_columns:
    raise ValueError(
        "Missing required scheduled-time audit columns: "
        f"{missing_schedule_columns}"
    )


def hhmm_to_minutes_after_midnight(value) -> float:
    """
    Convert valid raw HHMM values to minutes after local midnight.

    Rules
    -----
    - Missing values return NaN.
    - 0000 becomes 0.
    - 2400 becomes 1440, representing the following local midnight.
    - Other valid HHMM values become hour * 60 + minute.

    This function assumes the raw clock-time quality audit has already
    confirmed that non-missing values are valid HHMM or 2400.
    """
    if pd.isna(value):
        return np.nan

    hhmm = int(float(value))

    if hhmm == 2400:
        return 24 * 60

    hour = hhmm // 100
    minute = hhmm % 100

    return (hour * 60) + minute


# Full-file counters.
total_scheduled_rows = 0
missing_scheduled_elapsed_count = 0
arrival_clock_before_departure_count = 0
same_day_clock_difference_count = 0
overnight_adjusted_clock_difference_count = 0
unreconciled_schedule_count = 0

# Summary statistics for differences between raw-clock duration and
# reported scheduled elapsed time.
absolute_duration_difference_minutes = []

# Keep a small number of examples for later diagnostics.
schedule_audit_examples = []
MAX_SCHEDULE_EXAMPLES = 10

for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        usecols=SCHEDULE_AUDIT_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    total_scheduled_rows += len(chunk)

    departure_minutes = chunk["CRS_DEP_TIME"].apply(
        hhmm_to_minutes_after_midnight
    )

    arrival_minutes = chunk["CRS_ARR_TIME"].apply(
        hhmm_to_minutes_after_midnight
    )

    scheduled_elapsed_minutes = pd.to_numeric(
        chunk["CRS_ELAPSED_TIME"],
        errors="coerce",
    )

    missing_scheduled_elapsed_count += int(
        scheduled_elapsed_minutes.isna().sum()
    )

    # Arrival clock before departure clock is a useful raw signal:
    # it may reflect an overnight flight, a timezone difference, or both.
    arrival_before_departure = arrival_minutes < departure_minutes

    arrival_clock_before_departure_count += int(
        arrival_before_departure.sum()
    )

    # First calculate the simple raw-clock difference.
    raw_clock_duration = arrival_minutes - departure_minutes

    # If arrival clock is earlier than departure, add 24 hours.
    # This is only a diagnostic local-clock adjustment, not final UTC logic.
    adjusted_clock_duration = raw_clock_duration.where(
        ~arrival_before_departure,
        raw_clock_duration + (24 * 60),
    )

    # Rows with an arrival time at 2400 are inherently next-day local midnight.
    arrival_is_2400 = chunk["CRS_ARR_TIME"].eq(2400)

    adjusted_clock_duration = adjusted_clock_duration.where(
        ~arrival_is_2400,
        adjusted_clock_duration + (24 * 60),
    )

    same_day_clock_difference_count += int(
        (~arrival_before_departure & ~arrival_is_2400).sum()
    )

    overnight_adjusted_clock_difference_count += int(
        (arrival_before_departure | arrival_is_2400).sum()
    )

    # Compare raw-clock diagnostic duration with source scheduled elapsed time.
    # A large difference is expected for flights across timezones, because
    # local clock time cannot provide true duration without UTC normalisation.
    comparison_mask = scheduled_elapsed_minutes.notna()

    duration_difference = (
        adjusted_clock_duration[comparison_mask]
        - scheduled_elapsed_minutes[comparison_mask]
    )

    absolute_duration_difference = duration_difference.abs()

    # A tolerance of 30 minutes is used only for this preliminary audit.
    # It does not determine data validity or remove records.
    reconciled_mask = absolute_duration_difference <= 30

    unreconciled_schedule_count += int(
        (~reconciled_mask).sum()
    )

    absolute_duration_difference_minutes.extend(
        absolute_duration_difference.tolist()
    )

    # Keep a few examples of non-reconciled rows for later inspection.
    available_example_slots = (
        MAX_SCHEDULE_EXAMPLES - len(schedule_audit_examples)
    )

    if available_example_slots > 0:
        unreconciled_indices = absolute_duration_difference.index[
            ~reconciled_mask
        ][:available_example_slots]

        if len(unreconciled_indices) > 0:
            example_rows = chunk.loc[
                unreconciled_indices,
                [
                    "FL_DATE",
                    "ORIGIN",
                    "DEST",
                    "CRS_DEP_TIME",
                    "CRS_ARR_TIME",
                    "CRS_ELAPSED_TIME",
                ],
            ].copy()

            example_rows["raw_clock_duration_minutes"] = (
                adjusted_clock_duration.loc[unreconciled_indices]
            )

            example_rows["absolute_difference_minutes"] = (
                absolute_duration_difference.loc[unreconciled_indices]
            )

            schedule_audit_examples.extend(
                example_rows.to_dict(orient="records")
            )

    if chunk_number % 10 == 0:
        print(
            f"Audited scheduled-time consistency for "
            f"{total_scheduled_rows:,} rows."
        )

# Convert stored values to a NumPy array for stable summary statistics.
absolute_duration_difference_minutes = np.asarray(
    absolute_duration_difference_minutes,
    dtype=float,
)

schedule_duration_summary = {
    "median_absolute_difference_minutes": float(
        np.nanmedian(absolute_duration_difference_minutes)
    ),
    "p95_absolute_difference_minutes": float(
        np.nanpercentile(absolute_duration_difference_minutes, 95)
    ),
    "maximum_absolute_difference_minutes": float(
        np.nanmax(absolute_duration_difference_minutes)
    ),
}

schedule_consistency_summary = pd.DataFrame(
    [
        {
            "total_scheduled_rows": total_scheduled_rows,
            "missing_crs_elapsed_time": missing_scheduled_elapsed_count,
            "arrival_clock_before_departure": (
                arrival_clock_before_departure_count
            ),
            "same_day_raw_clock_difference": (
                same_day_clock_difference_count
            ),
            "overnight_adjusted_raw_clock_difference": (
                overnight_adjusted_clock_difference_count
            ),
            "unreconciled_within_30_minutes": (
                unreconciled_schedule_count
            ),
            **schedule_duration_summary,
        }
    ]
)

SCHEDULE_AUDIT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "scheduled_time_consistency_audit.csv"
)

schedule_consistency_summary.to_csv(
    SCHEDULE_AUDIT_OUTPUT_PATH,
    index=False,
)

print("\nScheduled-time consistency audit")
print("-" * 60)
print(f"Total scheduled rows: {total_scheduled_rows:,}")
print(
    "Missing CRS_ELAPSED_TIME: "
    f"{missing_scheduled_elapsed_count:,} "
    f"({missing_scheduled_elapsed_count / total_scheduled_rows:.2%})"
)
print(
    "Arrival clock earlier than departure clock: "
    f"{arrival_clock_before_departure_count:,} "
    f"({arrival_clock_before_departure_count / total_scheduled_rows:.2%})"
)
print(
    "Rows using same-day raw clock difference: "
    f"{same_day_clock_difference_count:,}"
)
print(
    "Rows requiring a local overnight adjustment: "
    f"{overnight_adjusted_clock_difference_count:,}"
)
print(
    "Rows not reconciled with CRS_ELAPSED_TIME within 30 minutes: "
    f"{unreconciled_schedule_count:,} "
    f"({unreconciled_schedule_count / total_scheduled_rows:.2%})"
)
print(
    "Median absolute duration difference: "
    f"{schedule_duration_summary['median_absolute_difference_minutes']:.1f} minutes"
)
print(
    "95th-percentile absolute duration difference: "
    f"{schedule_duration_summary['p95_absolute_difference_minutes']:.1f} minutes"
)
print(
    "Maximum absolute duration difference: "
    f"{schedule_duration_summary['maximum_absolute_difference_minutes']:.1f} minutes"
)

print("\nExamples requiring UTC-aware interpretation:")
for example in schedule_audit_examples:
    print(example)

print(f"\nSaved schedule-consistency audit: {SCHEDULE_AUDIT_OUTPUT_PATH}")

Audited scheduled-time consistency for 1,000,000 rows.
Audited scheduled-time consistency for 2,000,000 rows.
Audited scheduled-time consistency for 3,000,000 rows.

Scheduled-time consistency audit
------------------------------------------------------------
Total scheduled rows: 3,000,000
Missing CRS_ELAPSED_TIME: 14 (0.00%)
Arrival clock earlier than departure clock: 93,293 (3.11%)
Rows using same-day raw clock difference: 2,906,693
Rows requiring a local overnight adjustment: 93,307
Rows not reconciled with CRS_ELAPSED_TIME within 30 minutes: 1,424,977 (47.50%)
Median absolute duration difference: 0.0 minutes
95th-percentile absolute duration difference: 180.0 minutes
Maximum absolute duration difference: 1500.0 minutes

Examples requiring UTC-aware interpretation:
{'FL_DATE': '2022-11-19', 'ORIGIN': 'MSP', 'DEST': 'SEA', 'CRS_DEP_TIME': 2120, 'CRS_ARR_TIME': 2315, 'CRS_ELAPSED_TIME': 235.0, 'raw_clock_duration_minutes': 115, 'absolute_difference_minutes': 120.0}
{'FL_DATE': '2022-

In [10]:
# Build an auditable airport-coverage table from the raw flight CSV.
#
# This step uses only the existing Kaggle flight data.
# It does not introduce a new external dataset or create time zones yet.
#
# The output is a compact reference table containing:
# - airport IATA code
# - observed city name(s)
# - count of flights departing from the airport
# - count of flights arriving at the airport
# - total observed flight appearances
#
# It will become the controlled input for the airport timezone-mapping task.

AIRPORT_AUDIT_COLUMNS = [
    "ORIGIN",
    "ORIGIN_CITY",
    "DEST",
    "DEST_CITY",
]

missing_airport_columns = sorted(
    set(AIRPORT_AUDIT_COLUMNS) - raw_csv_columns
)

if missing_airport_columns:
    raise ValueError(
        "Missing required airport audit columns: "
        f"{missing_airport_columns}"
    )

# Store counts and observed city labels incrementally.
origin_flight_counts = Counter()
destination_flight_counts = Counter()
airport_city_names = {}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        CSV_PATH,
        usecols=AIRPORT_AUDIT_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    # Count origin appearances.
    origin_flight_counts.update(
        chunk["ORIGIN"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    # Count destination appearances.
    destination_flight_counts.update(
        chunk["DEST"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    # Capture the first non-null city name observed for each airport code.
    origin_city_pairs = (
        chunk[["ORIGIN", "ORIGIN_CITY"]]
        .dropna(subset=["ORIGIN"])
        .drop_duplicates()
    )

    for airport_code, city_name in origin_city_pairs.itertuples(
        index=False,
        name=None,
    ):
        airport_code = str(airport_code).strip()

        if airport_code not in airport_city_names and pd.notna(city_name):
            airport_city_names[airport_code] = str(city_name).strip()

    destination_city_pairs = (
        chunk[["DEST", "DEST_CITY"]]
        .dropna(subset=["DEST"])
        .drop_duplicates()
    )

    for airport_code, city_name in destination_city_pairs.itertuples(
        index=False,
        name=None,
    ):
        airport_code = str(airport_code).strip()

        if airport_code not in airport_city_names and pd.notna(city_name):
            airport_city_names[airport_code] = str(city_name).strip()

    if chunk_number % 10 == 0:
        print(
            f"Processed airport coverage for "
            f"{chunk_number * CHUNK_SIZE:,} rows."
        )

# Combine all airport codes seen in either origin or destination position.
all_airport_codes = sorted(
    set(origin_flight_counts) | set(destination_flight_counts)
)

airport_coverage = pd.DataFrame(
    {
        "airport_iata_code": all_airport_codes,
        "observed_city_name": [
            airport_city_names.get(code, pd.NA)
            for code in all_airport_codes
        ],
        "origin_flight_count": [
            origin_flight_counts.get(code, 0)
            for code in all_airport_codes
        ],
        "destination_flight_count": [
            destination_flight_counts.get(code, 0)
            for code in all_airport_codes
        ],
    }
)

airport_coverage["total_flight_appearances"] = (
    airport_coverage["origin_flight_count"]
    + airport_coverage["destination_flight_count"]
)

airport_coverage = airport_coverage.sort_values(
    by="total_flight_appearances",
    ascending=False,
).reset_index(drop=True)

# Write a small, Git-trackable reference output.
AIRPORT_COVERAGE_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "airport_coverage_audit.csv"
)

airport_coverage.to_csv(
    AIRPORT_COVERAGE_OUTPUT_PATH,
    index=False,
)

print("\nAirport coverage audit")
print("-" * 60)
print(f"Unique airport codes: {len(airport_coverage):,}")
print(
    "Airport codes with no recorded city label: "
    f"{airport_coverage['observed_city_name'].isna().sum():,}"
)
print(
    "Total origin-flight appearances: "
    f"{airport_coverage['origin_flight_count'].sum():,}"
)
print(
    "Total destination-flight appearances: "
    f"{airport_coverage['destination_flight_count'].sum():,}"
)

print("\nTop 20 airports by total flight appearances:")
display(airport_coverage.head(20))

print(f"\nSaved airport coverage audit: {AIRPORT_COVERAGE_OUTPUT_PATH}")

Processed airport coverage for 1,000,000 rows.
Processed airport coverage for 2,000,000 rows.
Processed airport coverage for 3,000,000 rows.

Airport coverage audit
------------------------------------------------------------
Unique airport codes: 380
Airport codes with no recorded city label: 0
Total origin-flight appearances: 3,000,000
Total destination-flight appearances: 3,000,000

Top 20 airports by total flight appearances:


,airport_iata_code,observed_city_name,origin_flight_count,destination_flight_count,total_flight_appearances
0,ATL,"Atlanta, GA",153556,153569,307125
1,DFW,"Dallas/Fort Worth, TX",130334,129770,260104
2,ORD,"Chicago, IL",122296,123334,245630
3,DEN,"Denver, CO",119919,119592,239511
4,CLT,"Charlotte, NC",94304,95413,189717
5,LAX,"Los Angeles, CA",85872,85621,171493
6,PHX,"Phoenix, AZ",74815,75605,150420
7,LAS,"Las Vegas, NV",73470,73462,146932
8,SEA,"Seattle, WA",70906,70832,141738
9,MCO,"Orlando, FL",63883,63818,127701



Saved airport coverage audit: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\airport_coverage_audit.csv


In [11]:
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

# Create the controlled airport-timezone mapping template.
#
# This is a project reference artefact, not a third operational dataset.
# It starts only from the airport codes and city names already present in the
# Kaggle flight dataset. We will populate and validate IANA timezones next.
#
# Required final fields:
# - airport_iata_code: airport identifier used in the raw source
# - observed_city_name: city label observed in the raw source
# - iana_timezone: timezone used for UTC normalisation, e.g. America/New_York
# - mapping_method: how the mapping was assigned and verified
# - mapping_status: pending, mapped, or requires_review
# - notes: exceptional cases or manual verification notes

airport_timezone_mapping = airport_coverage[
    [
        "airport_iata_code",
        "observed_city_name",
        "origin_flight_count",
        "destination_flight_count",
        "total_flight_appearances",
    ]
].copy()

# Sort by airport code so the mapping is stable and easy to review in Git.
airport_timezone_mapping = airport_timezone_mapping.sort_values(
    by="airport_iata_code",
    ascending=True,
).reset_index(drop=True)

# Initialise mapping fields. No timezones are guessed in this step.
airport_timezone_mapping["iana_timezone"] = pd.NA
airport_timezone_mapping["mapping_method"] = "Pending documented mapping"
airport_timezone_mapping["mapping_status"] = "pending"
airport_timezone_mapping["notes"] = pd.NA

# Save the initial mapping template as a small Git-trackable CSV.
AIRPORT_TIMEZONE_MAPPING_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "airport_timezone_mapping.csv"
)

airport_timezone_mapping.to_csv(
    AIRPORT_TIMEZONE_MAPPING_PATH,
    index=False,
)

print("Airport timezone mapping template")
print("-" * 60)
print(f"Airport rows requiring mapping: {len(airport_timezone_mapping):,}")
print(
    "Airport rows with an assigned IANA timezone: "
    f"{airport_timezone_mapping['iana_timezone'].notna().sum():,}"
)
print(
    "Airport rows still pending: "
    f"{airport_timezone_mapping['mapping_status'].eq('pending').sum():,}"
)
print(f"\nSaved mapping template:\n{AIRPORT_TIMEZONE_MAPPING_PATH}")

print("\nFirst 20 airport rows, sorted by IATA code:")
display(airport_timezone_mapping.head(20))

# Safety check: demonstrate that zoneinfo is available in the active
# Python environment. We will use this standard-library module later
# to create timezone-aware local timestamps and UTC timestamps.
example_timezone = "America/New_York"

try:
    ZoneInfo(example_timezone)
    print(
        f"\nPython zoneinfo validation succeeded for: {example_timezone}"
    )
except ZoneInfoNotFoundError as error:
    raise RuntimeError(
        "Python could not locate IANA timezone data. "
        "Install the tzdata package in your project environment before "
        "timestamp normalisation."
    ) from error

Airport timezone mapping template
------------------------------------------------------------
Airport rows requiring mapping: 380
Airport rows with an assigned IANA timezone: 0
Airport rows still pending: 380

Saved mapping template:
c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\airport_timezone_mapping.csv

First 20 airport rows, sorted by IATA code:


,airport_iata_code,observed_city_name,origin_flight_count,destination_flight_count,total_flight_appearances,iana_timezone,mapping_method,mapping_status,notes
0,ABE,"Allentown/Bethlehem/Easton, PA",2023,2044,4067,<NA>,Pending documented mapping,pending,<NA>
1,ABI,"Abilene, TX",901,943,1844,<NA>,Pending documented mapping,pending,<NA>
2,ABQ,"Albuquerque, NM",9284,9264,18548,<NA>,Pending documented mapping,pending,<NA>
3,ABR,"Aberdeen, SD",332,330,662,<NA>,Pending documented mapping,pending,<NA>
4,ABY,"Albany, GA",436,437,873,<NA>,Pending documented mapping,pending,<NA>
5,ACK,"Nantucket, MA",599,629,1228,<NA>,Pending documented mapping,pending,<NA>
6,ACT,"Waco, TX",719,675,1394,<NA>,Pending documented mapping,pending,<NA>
7,ACV,"Arcata/Eureka, CA",842,849,1691,<NA>,Pending documented mapping,pending,<NA>
8,ACY,"Atlantic City, NJ",1355,1350,2705,<NA>,Pending documented mapping,pending,<NA>
9,ADK,"Adak Island, AK",41,53,94,<NA>,Pending documented mapping,pending,<NA>



Python zoneinfo validation succeeded for: America/New_York


In [12]:
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

TEST_TIMEZONE = "America/New_York"

try:
    ZoneInfo(TEST_TIMEZONE)

    print(
        "ZoneInfo is working correctly. "
        f"IANA timezone available: {TEST_TIMEZONE}"
    )

except ZoneInfoNotFoundError:
    print(
        "ZoneInfo timezone data is not available in this environment. "
        "Install tzdata before continuing."
    )

ZoneInfo is working correctly. IANA timezone available: America/New_York


In [13]:
# Populate the airport-to-IANA-timezone mapping for the audited airport list.
#
# This mapping is intentionally explicit and version-controlled:
# - Every airport in the flight data must receive an IANA timezone.
# - The mapping is validated against Python's zoneinfo database.
# - Any airport missing from the mapping is written to a review list and
#   causes the cell to stop rather than silently assigning a timezone.
#
# Source basis:
# - IATA airport identifiers from the Kaggle flight data.
# - IANA timezone identifiers used by Python zoneinfo.
# - Airport timezone mapping requires documented manual verification.
#
# Note:
# Arizona does not observe daylight saving time, so Arizona airports use
# America/Phoenix rather than America/Denver.
# Hawaii uses Pacific/Honolulu.
# Alaska is split between America/Anchorage and America/Adak.
# U.S. territories use their respective IANA zones.

airport_timezone_lookup = {
    # Eastern Time
    "ABE": "America/New_York",
    "ABY": "America/New_York",
    "ACK": "America/New_York",
    "ALB": "America/New_York",
    "ATL": "America/New_York",
    "AVP": "America/New_York",
    "BDL": "America/New_York",
    "BGR": "America/New_York",
    "BHM": "America/Chicago",
    "BNA": "America/Chicago",
    "BOS": "America/New_York",
    "BQK": "America/New_York",
    "BTV": "America/New_York",
    "BUF": "America/New_York",
    "BWI": "America/New_York",
    "CAE": "America/New_York",
    "CAK": "America/New_York",
    "CHO": "America/New_York",
    "CHS": "America/New_York",
    "CID": "America/Chicago",
    "CLE": "America/New_York",
    "CLT": "America/New_York",
    "CMH": "America/New_York",
    "CRW": "America/New_York",
    "CVG": "America/New_York",
    "DAB": "America/New_York",
    "DAY": "America/New_York",
    "DCA": "America/New_York",
    "DRO": "America/Denver",
    "DTW": "America/New_York",
    "ECP": "America/Chicago",
    "ELM": "America/New_York",
    "ERI": "America/New_York",
    "EWR": "America/New_York",
    "FAY": "America/New_York",
    "FLL": "America/New_York",
    "FNT": "America/New_York",
    "FSD": "America/Chicago",
    "GNV": "America/New_York",
    "GRR": "America/New_York",
    "GSO": "America/New_York",
    "GSP": "America/New_York",
    "GTF": "America/Denver",
    "GTR": "America/Chicago",
    "HPN": "America/New_York",
    "HTS": "America/New_York",
    "IAD": "America/New_York",
    "ILM": "America/New_York",
    "IND": "America/Indiana/Indianapolis",
    "IPT": "America/New_York",
    "JAX": "America/New_York",
    "JFK": "America/New_York",
    "LAN": "America/New_York",
    "LGA": "America/New_York",
    "LEX": "America/New_York",
    "LIT": "America/Chicago",
    "MBS": "America/New_York",
    "MCI": "America/Chicago",
    "MCO": "America/New_York",
    "MDT": "America/New_York",
    "MGM": "America/Chicago",
    "MHT": "America/New_York",
    "MIA": "America/New_York",
    "MKE": "America/Chicago",
    "MLB": "America/New_York",
    "MOB": "America/Chicago",
    "MQT": "America/New_York",
    "MSN": "America/Chicago",
    "MYR": "America/New_York",
    "ORF": "America/New_York",
    "PBI": "America/New_York",
    "PIT": "America/New_York",
    "PNS": "America/Chicago",
    "PVD": "America/New_York",
    "PWM": "America/New_York",
    "RDU": "America/New_York",
    "RIC": "America/New_York",
    "ROA": "America/New_York",
    "ROC": "America/New_York",
    "RSW": "America/New_York",
    "SAV": "America/New_York",
    "SBN": "America/Indiana/Indianapolis",
    "SCE": "America/New_York",
    "SDF": "America/Kentucky/Louisville",
    "SHD": "America/New_York",
    "SJU": "America/Puerto_Rico",
    "SRQ": "America/New_York",
    "STL": "America/Chicago",
    "SYR": "America/New_York",
    "TLH": "America/New_York",
    "TPA": "America/New_York",
    "TRI": "America/New_York",
    "TTN": "America/New_York",
    "TYS": "America/New_York",
    "VPS": "America/Chicago",
    "XNA": "America/Chicago",

    # Central Time
    "ABI": "America/Chicago",
    "ACT": "America/Chicago",
    "AEX": "America/Chicago",
    "AMA": "America/Chicago",
    "AUS": "America/Chicago",
    "BIS": "America/Chicago",
    "BMI": "America/Chicago",
    "BRO": "America/Chicago",
    "BTR": "America/Chicago",
    "BTV": "America/New_York",
    "CDC": "America/Denver",
    "CHA": "America/New_York",
    "CMI": "America/Chicago",
    "CWA": "America/Chicago",
    "DAL": "America/Chicago",
    "DFW": "America/Chicago",
    "DLH": "America/Chicago",
    "DSM": "America/Chicago",
    "EAR": "America/Chicago",
    "EVV": "America/Indiana/Knox",
    "FAR": "America/Chicago",
    "FAT": "America/Los_Angeles",
    "FWA": "America/Indiana/Indianapolis",
    "GRB": "America/Chicago",
    "HOU": "America/Chicago",
    "HSV": "America/Chicago",
    "ICT": "America/Chicago",
    "JAN": "America/Chicago",
    "JLN": "America/Chicago",
    "LBB": "America/Chicago",
    "LCH": "America/Chicago",
    "LFT": "America/Chicago",
    "LNK": "America/Chicago",
    "LRD": "America/Chicago",
    "MAF": "America/Chicago",
    "MFE": "America/Chicago",
    "MHK": "America/Chicago",
    "MOT": "America/Chicago",
    "MSP": "America/Chicago",
    "MSY": "America/Chicago",
    "OKC": "America/Chicago",
    "OMA": "America/Chicago",
    "ORD": "America/Chicago",
    "PIA": "America/Chicago",
    "RST": "America/Chicago",
    "SAT": "America/Chicago",
    "SGF": "America/Chicago",
    "SHV": "America/Chicago",
    "SPS": "America/Chicago",
    "TUL": "America/Chicago",
    "TXK": "America/Chicago",
    "TYR": "America/Chicago",
    "UIN": "America/Chicago",
    "VCT": "America/Chicago",
    "YUM": "America/Phoenix",

    # Mountain Time
    "ABQ": "America/Denver",
    "ASE": "America/Denver",
    "BIL": "America/Denver",
    "BZN": "America/Denver",
    "CNY": "America/Denver",
    "COD": "America/Denver",
    "COS": "America/Denver",
    "CPR": "America/Denver",
    "DEN": "America/Denver",
    "DRO": "America/Denver",
    "EGE": "America/Denver",
    "FCA": "America/Denver",
    "GCC": "America/Denver",
    "GJT": "America/Denver",
    "HDN": "America/Denver",
    "IDA": "America/Boise",
    "JAC": "America/Denver",
    "LAR": "America/Denver",
    "MFR": "America/Los_Angeles",
    "MSO": "America/Denver",
    "PIH": "America/Boise",
    "PSC": "America/Los_Angeles",
    "PUB": "America/Denver",
    "RAP": "America/Denver",
    "RDM": "America/Los_Angeles",
    "RKS": "America/Denver",
    "SLC": "America/Denver",
    "SUN": "America/Boise",
    "TWF": "America/Boise",

    # Arizona: no daylight-saving clock change
    "AZA": "America/Phoenix",
    "FLG": "America/Phoenix",
    "GCN": "America/Phoenix",
    "INW": "America/Phoenix",
    "PHX": "America/Phoenix",
    "PRC": "America/Phoenix",
    "TUS": "America/Phoenix",
    "YUM": "America/Phoenix",

    # Pacific Time
    "ACV": "America/Los_Angeles",
    "BUR": "America/Los_Angeles",
    "BFL": "America/Los_Angeles",
    "BOI": "America/Boise",
    "BQN": "America/Puerto_Rico",
    "CLD": "America/Los_Angeles",
    "EUG": "America/Los_Angeles",
    "GEG": "America/Los_Angeles",
    "LAX": "America/Los_Angeles",
    "LGB": "America/Los_Angeles",
    "LIH": "Pacific/Honolulu",
    "MRY": "America/Los_Angeles",
    "OAK": "America/Los_Angeles",
    "OGG": "Pacific/Honolulu",
    "ONT": "America/Los_Angeles",
    "PDX": "America/Los_Angeles",
    "PSP": "America/Los_Angeles",
    "SAN": "America/Los_Angeles",
    "SBA": "America/Los_Angeles",
    "SCK": "America/Los_Angeles",
    "SFO": "America/Los_Angeles",
    "SJC": "America/Los_Angeles",
    "SMF": "America/Los_Angeles",
    "SNA": "America/Los_Angeles",
    "STS": "America/Los_Angeles",

    # Alaska
    "ADQ": "America/Anchorage",
    "ANC": "America/Anchorage",
    "BET": "America/Anchorage",
    "BRW": "America/Anchorage",
    "CDV": "America/Anchorage",
    "FAI": "America/Anchorage",
    "GST": "America/Juneau",
    "JNU": "America/Juneau",
    "KTN": "America/Juneau",
    "OTZ": "America/Anchorage",
    "SCC": "America/Anchorage",
    "SIT": "America/Juneau",
    "YAK": "America/Yakutat",

    # Hawaii
    "HNL": "Pacific/Honolulu",
    "ITO": "Pacific/Honolulu",
    "KOA": "Pacific/Honolulu",
    "LIH": "Pacific/Honolulu",
    "OGG": "Pacific/Honolulu",

    # U.S. territories and international airports commonly present in this sample
    "BQN": "America/Puerto_Rico",
    "GUM": "Pacific/Guam",
    "PSE": "America/Puerto_Rico",
    "SJU": "America/Puerto_Rico",
    "STT": "America/St_Thomas",
    "STX": "America/St_Thomas",
    "VPS": "America/Chicago",
}

# Apply the explicit mapping to the airport coverage table.
airport_timezone_mapping["iana_timezone"] = (
    airport_timezone_mapping["airport_iata_code"]
    .map(airport_timezone_lookup)
)

# Flag every mapped airport as pending validation first.
airport_timezone_mapping["mapping_status"] = np.where(
    airport_timezone_mapping["iana_timezone"].notna(),
    "mapped_pending_validation",
    "requires_review",
)

airport_timezone_mapping["mapping_method"] = np.where(
    airport_timezone_mapping["iana_timezone"].notna(),
    "Explicit IATA-to-IANA mapping; ZoneInfo validated",
    "No mapping available; manual review required",
)

airport_timezone_mapping["notes"] = np.where(
    airport_timezone_mapping["airport_iata_code"].isin(
        ["PHX", "TUS", "FLG", "GCN", "INW", "PRC", "YUM", "AZA"]
    ),
    "Arizona airport: America/Phoenix used because Arizona does not observe DST.",
    pd.NA,
)

# Validate every assigned timezone using Python's standard zoneinfo database.
invalid_iana_timezones = []

for timezone_name in (
    airport_timezone_mapping["iana_timezone"]
    .dropna()
    .unique()
):
    try:
        ZoneInfo(timezone_name)
    except ZoneInfoNotFoundError:
        invalid_iana_timezones.append(timezone_name)

if invalid_iana_timezones:
    raise ValueError(
        "The mapping contains invalid IANA timezone names: "
        f"{sorted(invalid_iana_timezones)}"
    )

# Identify airport codes that were present in the flight dataset but are absent
# from the explicit mapping. Save them for controlled manual review.
unmapped_airports = airport_timezone_mapping.loc[
    airport_timezone_mapping["iana_timezone"].isna(),
    [
        "airport_iata_code",
        "observed_city_name",
        "total_flight_appearances",
    ],
].copy()

UNMAPPED_AIRPORTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "airport_timezone_mapping_requires_review.csv"
)

unmapped_airports.to_csv(
    UNMAPPED_AIRPORTS_PATH,
    index=False,
)

# Do not claim the mapping is complete if even one airport is unresolved.
if not unmapped_airports.empty:
    print("Airport timezone mapping is incomplete.")
    print(
        f"Airports requiring manual review: {len(unmapped_airports):,}"
    )

    display(unmapped_airports)

    print(
        "\nA review file was saved. Do not proceed to timestamp normalisation "
        "until every airport is resolved."
    )
else:
    airport_timezone_mapping["mapping_status"] = "mapped_validated"

    airport_timezone_mapping.to_csv(
        AIRPORT_TIMEZONE_MAPPING_PATH,
        index=False,
    )

    print("Airport timezone mapping completed successfully.")
    print(
        f"Mapped airport rows: {len(airport_timezone_mapping):,}"
    )
    print(
        "Unique IANA timezones used: "
        f"{airport_timezone_mapping['iana_timezone'].nunique():,}"
    )
    print(
        f"Saved validated mapping: {AIRPORT_TIMEZONE_MAPPING_PATH}"
    )

    print("\nTimezone mapping preview:")
    display(
        airport_timezone_mapping[
            [
                "airport_iata_code",
                "observed_city_name",
                "iana_timezone",
                "mapping_status",
                "notes",
            ]
        ].head(25)
    )

Airport timezone mapping is incomplete.
Airports requiring manual review: 153


,airport_iata_code,observed_city_name,total_flight_appearances
3,ABR,"Aberdeen, SD",662
8,ACY,"Atlantic City, NJ",2705
9,ADK,"Adak Island, AK",94
12,AGS,"Augusta, GA",3671
13,AKN,"King Salmon, AK",277
...,...,...,...
371,VLD,"Valdosta, GA",954
373,WRG,"Wrangell, AK",667
374,WYS,"West Yellowstone, MT",299
376,XWA,"Williston, ND",890



A review file was saved. Do not proceed to timestamp normalisation until every airport is resolved.


In [14]:
# Display every unresolved airport in a compact, review-friendly table.
#
# This does not change any project data. It lets us see the exact 153
# airport codes that must be assigned a valid IANA timezone before UTC
# timestamp reconstruction can begin.

unmapped_airports_sorted = unmapped_airports.sort_values(
    by=["total_flight_appearances", "airport_iata_code"],
    ascending=[False, True],
).reset_index(drop=True)

print("Complete list of airports requiring timezone mapping")
print("-" * 60)
print(f"Total unresolved airports: {len(unmapped_airports_sorted):,}")

display(unmapped_airports_sorted)

print("\nTop 30 unresolved airports by flight appearances:")
display(unmapped_airports_sorted.head(30))

Complete list of airports requiring timezone mapping
------------------------------------------------------------
Total unresolved airports: 153


,airport_iata_code,observed_city_name,total_flight_appearances
0,LAS,"Las Vegas, NV",146932
1,SEA,"Seattle, WA",141738
2,IAH,"Houston, TX",124743
3,PHL,"Philadelphia, PA",85519
4,MDW,"Chicago, IL",69825
...,...,...,...
148,OGD,"Ogden, UT",49
149,ART,"Watertown, NY",47
150,CDB,"Cold Bay, AK",45
151,ILG,"Wilmington, DE",30



Top 30 unresolved airports by flight appearances:


,airport_iata_code,observed_city_name,total_flight_appearances
0,LAS,"Las Vegas, NV",146932
1,SEA,"Seattle, WA",141738
2,IAH,"Houston, TX",124743
3,PHL,"Philadelphia, PA",85519
4,MDW,"Chicago, IL",69825
5,MEM,"Memphis, TN",19837
6,RNO,"Reno, NV",17681
7,ELP,"El Paso, TX",14273
8,SFB,"Sanford, FL",9057
9,AVL,"Asheville, NC",8382


In [15]:
import airportsdata

print(
    "airportsdata imported successfully. "
    f"Installed package location: {airportsdata.__file__}"
)

airportsdata imported successfully. Installed package location: c:\Users\Softmgmt\anaconda3\Lib\site-packages\airportsdata\__init__.py


In [16]:
import importlib.metadata as importlib_metadata
import airportsdata

# Load the airport reference database indexed by IATA airport code.
#
# This database is used only to construct the project-owned airport reference
# mapping needed for local-time interpretation and UTC normalisation.
# The final mapping CSV remains the auditable project artefact.
airport_reference_by_iata = airportsdata.load("IATA")

# Record package version for reproducibility.
try:
    airportsdata_version = importlib_metadata.version("airportsdata")
except importlib_metadata.PackageNotFoundError:
    airportsdata_version = "unknown"

print("Airport reference source loaded")
print("-" * 60)
print(f"Reference package: airportsdata")
print(f"Reference package version: {airportsdata_version}")
print(f"Reference records indexed by IATA code: {len(airport_reference_by_iata):,}")

# Start from the audited airport list, not from all airports in the package.
validated_airport_mapping = airport_coverage[
    [
        "airport_iata_code",
        "observed_city_name",
        "origin_flight_count",
        "destination_flight_count",
        "total_flight_appearances",
    ]
].copy()

validated_airport_mapping = validated_airport_mapping.sort_values(
    by="airport_iata_code",
    ascending=True,
).reset_index(drop=True)

# Look up each IATA code in the reference database.
reference_records = validated_airport_mapping["airport_iata_code"].map(
    airport_reference_by_iata
)

# Extract reference fields carefully. An unmatched IATA code produces None.
validated_airport_mapping["reference_airport_name"] = reference_records.map(
    lambda record: record.get("name") if isinstance(record, dict) else pd.NA
)

validated_airport_mapping["reference_city"] = reference_records.map(
    lambda record: record.get("city") if isinstance(record, dict) else pd.NA
)

validated_airport_mapping["reference_country"] = reference_records.map(
    lambda record: record.get("country") if isinstance(record, dict) else pd.NA
)

validated_airport_mapping["latitude_deg"] = reference_records.map(
    lambda record: record.get("lat") if isinstance(record, dict) else pd.NA
)

validated_airport_mapping["longitude_deg"] = reference_records.map(
    lambda record: record.get("lon") if isinstance(record, dict) else pd.NA
)

validated_airport_mapping["iana_timezone"] = reference_records.map(
    lambda record: record.get("tz") if isinstance(record, dict) else pd.NA
)

# Create explicit match and review flags.
validated_airport_mapping["reference_match_found"] = reference_records.notna()

validated_airport_mapping["mapping_method"] = np.where(
    validated_airport_mapping["reference_match_found"],
    (
        "Matched by IATA code using airportsdata "
        f"version {airportsdata_version}"
    ),
    (
        "No airportsdata IATA match; manual review required"
    ),
)

validated_airport_mapping["mapping_status"] = np.where(
    validated_airport_mapping["reference_match_found"]
    & validated_airport_mapping["iana_timezone"].notna(),
    "matched_pending_timezone_validation",
    "requires_review",
)

validated_airport_mapping["notes"] = pd.NA

# Validate every mapped timezone identifier using Python's zoneinfo database.
timezone_validation_errors = []

for timezone_name in (
    validated_airport_mapping["iana_timezone"]
    .dropna()
    .astype(str)
    .unique()
):
    try:
        ZoneInfo(timezone_name)
    except ZoneInfoNotFoundError:
        timezone_validation_errors.append(timezone_name)

# Mark any reference records that do not contain a valid ZoneInfo timezone.
if timezone_validation_errors:
    invalid_timezone_mask = validated_airport_mapping[
        "iana_timezone"
    ].isin(timezone_validation_errors)

    validated_airport_mapping.loc[
        invalid_timezone_mask,
        "mapping_status",
    ] = "requires_review"

    validated_airport_mapping.loc[
        invalid_timezone_mask,
        "notes",
    ] = (
        "Reference timezone was not recognised by Python ZoneInfo; "
        "manual review required."
    )

# Build the review list: unmatched airports, missing timezones, or invalid zones.
airport_mapping_review = validated_airport_mapping.loc[
    validated_airport_mapping["mapping_status"].eq("requires_review"),
    [
        "airport_iata_code",
        "observed_city_name",
        "total_flight_appearances",
        "reference_airport_name",
        "reference_city",
        "reference_country",
        "latitude_deg",
        "longitude_deg",
        "iana_timezone",
        "notes",
    ],
].copy()

AIRPORT_MAPPING_REVIEW_PATH = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "airport_timezone_mapping_requires_review.csv"
)

airport_mapping_review.to_csv(
    AIRPORT_MAPPING_REVIEW_PATH,
    index=False,
)

# Only label all mapped rows as validated after ZoneInfo verification.
validated_timezone_mask = validated_airport_mapping[
    "mapping_status"
].eq("matched_pending_timezone_validation")

validated_airport_mapping.loc[
    validated_timezone_mask,
    "mapping_status",
] = "mapped_validated"

# Save the complete controlled mapping as the project reference artefact.
validated_airport_mapping.to_csv(
    AIRPORT_TIMEZONE_MAPPING_PATH,
    index=False,
)

# Print audit results.
matched_airport_count = int(
    validated_airport_mapping["reference_match_found"].sum()
)

validated_timezone_count = int(
    validated_airport_mapping["mapping_status"].eq(
        "mapped_validated"
    ).sum()
)

requires_review_count = int(
    validated_airport_mapping["mapping_status"].eq(
        "requires_review"
    ).sum()
)

print("\nAirport timezone-reference mapping audit")
print("-" * 60)
print(f"Audited airport codes: {len(validated_airport_mapping):,}")
print(f"Reference matches by IATA code: {matched_airport_count:,}")
print(f"ZoneInfo-validated timezone mappings: {validated_timezone_count:,}")
print(f"Airports requiring manual review: {requires_review_count:,}")
print(f"Invalid timezone names found: {len(timezone_validation_errors):,}")
print(f"\nSaved complete mapping: {AIRPORT_TIMEZONE_MAPPING_PATH}")
print(f"Saved review list: {AIRPORT_MAPPING_REVIEW_PATH}")

if requires_review_count > 0:
    print("\nAirports requiring manual review:")
    display(
        airport_mapping_review.sort_values(
            by=["total_flight_appearances", "airport_iata_code"],
            ascending=[False, True],
        )
    )
else:
    print("\nAll audited airports have a ZoneInfo-validated IANA timezone.")
    print("\nMapping preview:")
    display(
        validated_airport_mapping[
            [
                "airport_iata_code",
                "observed_city_name",
                "reference_airport_name",
                "reference_city",
                "latitude_deg",
                "longitude_deg",
                "iana_timezone",
                "mapping_status",
            ]
        ].head(20)
    )

Airport reference source loaded
------------------------------------------------------------
Reference package: airportsdata
Reference package version: 20260905
Reference records indexed by IATA code: 7,884

Airport timezone-reference mapping audit
------------------------------------------------------------
Audited airport codes: 380
Reference matches by IATA code: 379
ZoneInfo-validated timezone mappings: 379
Airports requiring manual review: 1
Invalid timezone names found: 0

Saved complete mapping: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\airport_timezone_mapping.csv
Saved review list: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\airport_timezone_mapping_requires_review.csv

Airports requiring manual review:


,airport_iata_code,observed_city_name,total_flight_appearances,reference_airport_name,reference_city,reference_country,latitude_deg,longitude_deg,iana_timezone,notes
183,ISN,"Williston, ND",122,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [17]:
# Resolve the single unmatched airport through a documented manual decision.
#
# ISN is Sloulin Field International Airport in Williston, North Dakota.
# America/North_Dakota/Beulah is used because it represents the relevant
# North Dakota Central Time and daylight-saving-time rules.
#
# This controlled exception remains visible in the project-owned mapping CSV.

MANUAL_AIRPORT_TIMEZONE_OVERRIDES = {
    "ISN": {
        "iana_timezone": "America/North_Dakota/Beulah",
        "reference_airport_name": "Sloulin Field International Airport",
        "reference_city": "Williston",
        "reference_country": "US",
        "mapping_method": (
            "Manual IATA-to-IANA mapping after airportsdata no-match; "
            "verified against airport/timezone reference"
        ),
        "notes": (
            "ISN / Sloulin Field International Airport, Williston, North Dakota. "
            "Manual mapping to America/North_Dakota/Beulah."
        ),
    }
}

# Confirm that the only unresolved airport is covered by our controlled override.
unresolved_codes = set(
    validated_airport_mapping.loc[
        validated_airport_mapping["mapping_status"].eq("requires_review"),
        "airport_iata_code",
    ]
)

override_codes = set(MANUAL_AIRPORT_TIMEZONE_OVERRIDES)

uncovered_unresolved_codes = unresolved_codes - override_codes

if uncovered_unresolved_codes:
    raise ValueError(
        "The following unresolved airports have no approved manual override: "
        f"{sorted(uncovered_unresolved_codes)}"
    )

# Apply each approved manual mapping.
for airport_code, override in MANUAL_AIRPORT_TIMEZONE_OVERRIDES.items():
    airport_mask = validated_airport_mapping["airport_iata_code"].eq(
        airport_code
    )

    if airport_mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one mapping row for {airport_code}, "
            f"but found {airport_mask.sum()}."
        )

    timezone_name = override["iana_timezone"]

    # Validate the IANA name before writing it into the final mapping.
    try:
        ZoneInfo(timezone_name)
    except ZoneInfoNotFoundError as error:
        raise ValueError(
            f"Manual timezone override is not recognised by ZoneInfo: "
            f"{timezone_name}"
        ) from error

    validated_airport_mapping.loc[
        airport_mask,
        "reference_airport_name",
    ] = override["reference_airport_name"]

    validated_airport_mapping.loc[
        airport_mask,
        "reference_city",
    ] = override["reference_city"]

    validated_airport_mapping.loc[
        airport_mask,
        "reference_country",
    ] = override["reference_country"]

    validated_airport_mapping.loc[
        airport_mask,
        "iana_timezone",
    ] = timezone_name

    validated_airport_mapping.loc[
        airport_mask,
        "reference_match_found",
    ] = False

    validated_airport_mapping.loc[
        airport_mask,
        "mapping_method",
    ] = override["mapping_method"]

    validated_airport_mapping.loc[
        airport_mask,
        "mapping_status",
    ] = "mapped_validated_manual_override"

    validated_airport_mapping.loc[
        airport_mask,
        "notes",
    ] = override["notes"]

# Final completeness and validity checks.
remaining_unmapped_airports = validated_airport_mapping.loc[
    validated_airport_mapping["iana_timezone"].isna(),
    "airport_iata_code",
].tolist()

if remaining_unmapped_airports:
    raise ValueError(
        "Airport timezone mapping remains incomplete: "
        f"{remaining_unmapped_airports}"
    )

invalid_final_timezones = []

for timezone_name in validated_airport_mapping["iana_timezone"].unique():
    try:
        ZoneInfo(timezone_name)
    except ZoneInfoNotFoundError:
        invalid_final_timezones.append(timezone_name)

if invalid_final_timezones:
    raise ValueError(
        "Final mapping contains invalid IANA timezone names: "
        f"{sorted(invalid_final_timezones)}"
    )

# Save the complete final mapping.
validated_airport_mapping.to_csv(
    AIRPORT_TIMEZONE_MAPPING_PATH,
    index=False,
)

# Replace the prior review file with an empty table that preserves its schema,
# proving that no airport remains unresolved.
empty_review_table = airport_mapping_review.iloc[0:0].copy()

empty_review_table.to_csv(
    AIRPORT_MAPPING_REVIEW_PATH,
    index=False,
)

print("Final airport-timezone mapping validation")
print("-" * 60)
print(f"Total audited airports: {len(validated_airport_mapping):,}")
print(
    "Airports with valid IANA timezone mappings: "
    f"{validated_airport_mapping['iana_timezone'].notna().sum():,}"
)
print(
    "Airports remaining unresolved: "
    f"{validated_airport_mapping['iana_timezone'].isna().sum():,}"
)
print(
    "Manual mapping overrides used: "
    f"{validated_airport_mapping['mapping_status'].eq('mapped_validated_manual_override').sum():,}"
)
print(
    "Invalid final IANA timezone names: "
    f"{len(invalid_final_timezones):,}"
)

print("\nISN mapping record:")
display(
    validated_airport_mapping.loc[
        validated_airport_mapping["airport_iata_code"].eq("ISN"),
        [
            "airport_iata_code",
            "observed_city_name",
            "reference_airport_name",
            "reference_city",
            "reference_country",
            "iana_timezone",
            "mapping_method",
            "mapping_status",
            "notes",
        ],
    ]
)

print(f"\nSaved final mapping: {AIRPORT_TIMEZONE_MAPPING_PATH}")
print(f"Saved empty review list: {AIRPORT_MAPPING_REVIEW_PATH}")

Final airport-timezone mapping validation
------------------------------------------------------------
Total audited airports: 380
Airports with valid IANA timezone mappings: 380
Airports remaining unresolved: 0
Manual mapping overrides used: 1
Invalid final IANA timezone names: 0

ISN mapping record:


,airport_iata_code,observed_city_name,reference_airport_name,reference_city,reference_country,iana_timezone,mapping_method,mapping_status,notes
183,ISN,"Williston, ND",Sloulin Field International Airport,Williston,US,America/North_Dakota/Beulah,Manual IATA-to-IANA mapping after airportsdata...,mapped_validated_manual_override,"ISN / Sloulin Field International Airport, Wil..."



Saved final mapping: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\airport_timezone_mapping.csv
Saved empty review list: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\data\external\airport_timezone_mapping_requires_review.csv


In [2]:
from pathlib import Path
import sys

# This notebook is stored inside the project's notebooks/ folder.
# Its parent directory is the project root.
PROJECT_ROOT = Path.cwd().parent

SRC_DIRECTORY = PROJECT_ROOT / "src"

print(f"Notebook working directory: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIRECTORY}")

if not SRC_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Project source directory was not found: {SRC_DIRECTORY}"
    )

if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

print("Project src/ directory added to the Python import path.")

Notebook working directory: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\notebooks
Project root: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence
Source directory: c:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Airline Network Disruption Intelligence\src
Project src/ directory added to the Python import path.


In [4]:
import pandas as pd
from airline_disruption.timestamps.local_to_utc import (
    convert_local_hhmm_to_utc,
)

timestamp_examples = [
    {
        "case": "Normal Eastern-time departure",
        "local_date": "2023-06-15",
        "raw_hhmm": 930,
        "iana_timezone": "America/New_York",
    },
    {
        "case": "Short HHMM value",
        "local_date": "2023-06-15",
        "raw_hhmm": 5,
        "iana_timezone": "America/Chicago",
    },
    {
        "case": "2400 rollover",
        "local_date": "2023-06-15",
        "raw_hhmm": 2400,
        "iana_timezone": "America/Los_Angeles",
    },
    {
        "case": "Spring DST nonexistent local time",
        "local_date": "2023-03-12",
        "raw_hhmm": 230,
        "iana_timezone": "America/New_York",
    },
    {
        "case": "Autumn DST ambiguous local time",
        "local_date": "2023-11-05",
        "raw_hhmm": 130,
        "iana_timezone": "America/New_York",
    },
]

timestamp_test_results = []

for example in timestamp_examples:
    result = convert_local_hhmm_to_utc(
        local_date=example["local_date"],
        raw_hhmm=example["raw_hhmm"],
        iana_timezone=example["iana_timezone"],
    )

    timestamp_test_results.append(
        {
            "case": example["case"],
            "local_date": example["local_date"],
            "raw_hhmm": example["raw_hhmm"],
            "iana_timezone": example["iana_timezone"],
            "local_timestamp": result.local_timestamp,
            "utc_timestamp": result.utc_timestamp,
            "date_rollover_days": result.date_rollover_days,
            "input_status": result.input_status,
        }
    )

timestamp_test_results_df = pd.DataFrame(timestamp_test_results)

display(timestamp_test_results_df)

,case,local_date,raw_hhmm,iana_timezone,local_timestamp,utc_timestamp,date_rollover_days,input_status
0,Normal Eastern-time departure,2023-06-15,930,America/New_York,2023-06-15 09:30:00-04:00,2023-06-15 13:30:00+00:00,0,valid
1,Short HHMM value,2023-06-15,5,America/Chicago,2023-06-15 00:05:00-05:00,2023-06-15 05:05:00+00:00,0,valid
2,2400 rollover,2023-06-15,2400,America/Los_Angeles,2023-06-16 00:00:00-07:00,2023-06-16 07:00:00+00:00,1,valid
3,Spring DST nonexistent local time,2023-03-12,230,America/New_York,NaT,NaT,0,timestamp_conversion_error
4,Autumn DST ambiguous local time,2023-11-05,130,America/New_York,NaT,NaT,0,ambiguous_local_time


In [5]:
import importlib
import airline_disruption.timestamps.local_to_utc as local_to_utc

importlib.reload(local_to_utc)

spring_dst_result = local_to_utc.convert_local_hhmm_to_utc(
    local_date="2023-03-12",
    raw_hhmm=230,
    iana_timezone="America/New_York",
)

autumn_dst_result = local_to_utc.convert_local_hhmm_to_utc(
    local_date="2023-11-05",
    raw_hhmm=130,
    iana_timezone="America/New_York",
)

print(
    "Spring DST status:",
    spring_dst_result.input_status,
)
print(
    "Autumn DST status:",
    autumn_dst_result.input_status,
)

Spring DST status: nonexistent_local_time
Autumn DST status: ambiguous_local_time


In [6]:
import pyarrow
import pyarrow.parquet as pq

print(f"PyArrow version: {pyarrow.__version__}")
print("Parquet support is available.")

PyArrow version: 19.0.0
Parquet support is available.


In [7]:
from pathlib import Path
import pandas as pd

# Re-create the project root in case the notebook kernel was restarted.
PROJECT_ROOT = Path.cwd().parent

SAMPLE_PARQUET_PATH = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "scheduled_timestamp_normalisation_sample.parquet"
)

if not SAMPLE_PARQUET_PATH.exists():
    raise FileNotFoundError(
        f"Sample Parquet file was not found: {SAMPLE_PARQUET_PATH}"
    )

sample_normalised_flights = pd.read_parquet(
    SAMPLE_PARQUET_PATH,
    engine="pyarrow",
)

TIMESTAMP_INSPECTION_COLUMNS = [
    "source_row_number",
    "flight_date_local",
    "origin_airport",
    "destination_airport",
    "origin_iana_timezone",
    "destination_iana_timezone",
    "scheduled_departure_time_local_hhmm",
    "scheduled_departure_local",
    "scheduled_departure_utc",
    "scheduled_arrival_time_local_hhmm",
    "scheduled_arrival_local",
    "scheduled_arrival_utc",
    "scheduled_arrival_date_offset_days",
    "scheduled_elapsed_minutes",
    "scheduled_duration_utc_minutes",
    "scheduled_duration_difference_minutes",
    "prediction_timestamp_utc",
    "scheduled_departure_timestamp_status",
    "scheduled_arrival_reconciliation_status",
]

missing_inspection_columns = sorted(
    set(TIMESTAMP_INSPECTION_COLUMNS)
    - set(sample_normalised_flights.columns)
)

if missing_inspection_columns:
    raise ValueError(
        "Expected timestamp inspection columns are missing: "
        f"{missing_inspection_columns}"
    )

print("Sample timestamp-normalisation inspection")
print("-" * 60)
print(f"Rows loaded: {len(sample_normalised_flights):,}")
print(f"Columns available: {len(sample_normalised_flights.columns):,}")

print("\nFirst 10 timestamp-normalised flights:")
display(
    sample_normalised_flights[
        TIMESTAMP_INSPECTION_COLUMNS
    ].head(10)
)

print("\nTimestamp status counts:")
print(
    sample_normalised_flights[
        "scheduled_departure_timestamp_status"
    ].value_counts(
        dropna=False
    )
)

print("\nArrival reconciliation status counts:")
print(
    sample_normalised_flights[
        "scheduled_arrival_reconciliation_status"
    ].value_counts(
        dropna=False
    )
)

print("\nUTC ordering checks:")
print(
    "Prediction timestamps before scheduled departures: "
    f"{(
        sample_normalised_flights['prediction_timestamp_utc']
        < sample_normalised_flights['scheduled_departure_utc']
    ).sum():,}"
)
print(
    "Scheduled arrivals on or after scheduled departures: "
    f"{(
        sample_normalised_flights['scheduled_arrival_utc']
        >= sample_normalised_flights['scheduled_departure_utc']
    ).sum():,}"
)
print(
    "Rows with zero scheduled-duration difference: "
    f"{(
        sample_normalised_flights[
            'scheduled_duration_difference_minutes'
        ]
        == 0
    ).sum():,}"
)

Sample timestamp-normalisation inspection
------------------------------------------------------------
Rows loaded: 1,000
Columns available: 54

First 10 timestamp-normalised flights:


,source_row_number,flight_date_local,origin_airport,destination_airport,origin_iana_timezone,destination_iana_timezone,scheduled_departure_time_local_hhmm,scheduled_departure_local,scheduled_departure_utc,scheduled_arrival_time_local_hhmm,scheduled_arrival_local,scheduled_arrival_utc,scheduled_arrival_date_offset_days,scheduled_elapsed_minutes,scheduled_duration_utc_minutes,scheduled_duration_difference_minutes,prediction_timestamp_utc,scheduled_departure_timestamp_status,scheduled_arrival_reconciliation_status
0,0,2019-01-09,FLL,EWR,America/New_York,America/New_York,1155,2019-01-09 11:55:00-05:00,2019-01-09 16:55:00+00:00,1501,2019-01-09 15:01:00-05:00,2019-01-09 20:01:00+00:00,0,186.0,186.0,0.0,2019-01-09 14:55:00+00:00,valid,reconciled
1,1,2022-11-19,MSP,SEA,America/Chicago,America/Los_Angeles,2120,2022-11-19 22:20:00-05:00,2022-11-20 03:20:00+00:00,2315,2022-11-20 02:15:00-05:00,2022-11-20 07:15:00+00:00,0,235.0,235.0,0.0,2022-11-20 01:20:00+00:00,valid,reconciled
2,2,2022-07-22,DEN,MSP,America/Denver,America/Chicago,954,2022-07-22 11:54:00-04:00,2022-07-22 15:54:00+00:00,1252,2022-07-22 13:52:00-04:00,2022-07-22 17:52:00+00:00,0,118.0,118.0,0.0,2022-07-22 13:54:00+00:00,valid,reconciled
3,3,2023-03-06,MSP,SFO,America/Chicago,America/Los_Angeles,1609,2023-03-06 17:09:00-05:00,2023-03-06 22:09:00+00:00,1829,2023-03-06 21:29:00-05:00,2023-03-07 02:29:00+00:00,0,260.0,260.0,0.0,2023-03-06 20:09:00+00:00,valid,reconciled
4,4,2020-02-23,MCO,DFW,America/New_York,America/Chicago,1840,2020-02-23 18:40:00-05:00,2020-02-23 23:40:00+00:00,2041,2020-02-23 21:41:00-05:00,2020-02-24 02:41:00+00:00,0,181.0,181.0,0.0,2020-02-23 21:40:00+00:00,valid,reconciled
5,5,2019-07-31,DAL,OKC,America/Chicago,America/Chicago,1010,2019-07-31 11:10:00-04:00,2019-07-31 15:10:00+00:00,1110,2019-07-31 12:10:00-04:00,2019-07-31 16:10:00+00:00,0,60.0,60.0,0.0,2019-07-31 13:10:00+00:00,valid,reconciled
6,6,2023-06-11,DCA,BOS,America/New_York,America/New_York,1010,2023-06-11 10:10:00-04:00,2023-06-11 14:10:00+00:00,1159,2023-06-11 11:59:00-04:00,2023-06-11 15:59:00+00:00,0,109.0,109.0,0.0,2023-06-11 12:10:00+00:00,valid,reconciled
7,7,2019-07-08,HSV,DCA,America/Chicago,America/New_York,1643,2019-07-08 17:43:00-04:00,2019-07-08 21:43:00+00:00,1945,2019-07-08 19:45:00-04:00,2019-07-08 23:45:00+00:00,0,122.0,122.0,0.0,2019-07-08 19:43:00+00:00,valid,reconciled
8,8,2023-02-12,IAH,LAX,America/Chicago,America/Los_Angeles,530,2023-02-12 06:30:00-05:00,2023-02-12 11:30:00+00:00,717,2023-02-12 10:17:00-05:00,2023-02-12 15:17:00+00:00,0,227.0,227.0,0.0,2023-02-12 09:30:00+00:00,valid,reconciled
9,9,2020-08-22,SEA,FAI,America/Los_Angeles,America/Anchorage,2125,2020-08-23 00:25:00-04:00,2020-08-23 04:25:00+00:00,2355,2020-08-23 03:55:00-04:00,2020-08-23 07:55:00+00:00,0,210.0,210.0,0.0,2020-08-23 02:25:00+00:00,valid,reconciled



Timestamp status counts:
scheduled_departure_timestamp_status
valid    1000
Name: count, dtype: int64

Arrival reconciliation status counts:
scheduled_arrival_reconciliation_status
reconciled    1000
Name: count, dtype: int64

UTC ordering checks:
Prediction timestamps before scheduled departures: 1,000
Scheduled arrivals on or after scheduled departures: 1,000
Rows with zero scheduled-duration difference: 1,000


In [8]:
sample_normalised_flights = pd.read_parquet(
    SAMPLE_PARQUET_PATH,
    engine="pyarrow",
)

print("Sample dtype verification")
print("-" * 60)
print(
    "flight_date_local dtype: "
    f"{sample_normalised_flights['flight_date_local'].dtype}"
)
print(
    "scheduled_departure_utc dtype: "
    f"{sample_normalised_flights['scheduled_departure_utc'].dtype}"
)
print(
    "scheduled_arrival_utc dtype: "
    f"{sample_normalised_flights['scheduled_arrival_utc'].dtype}"
)
print(
    "prediction_timestamp_utc dtype: "
    f"{sample_normalised_flights['prediction_timestamp_utc'].dtype}"
)

print("\nFirst five source local flight dates:")
print(
    sample_normalised_flights[
        "flight_date_local"
    ].head().tolist()
)

Sample dtype verification
------------------------------------------------------------
flight_date_local dtype: datetime64[ns]
scheduled_departure_utc dtype: datetime64[ns, UTC]
scheduled_arrival_utc dtype: datetime64[ns, UTC]
prediction_timestamp_utc dtype: datetime64[ns, UTC]

First five source local flight dates:
[Timestamp('2019-01-09 00:00:00'), Timestamp('2022-11-19 00:00:00'), Timestamp('2022-07-22 00:00:00'), Timestamp('2023-03-06 00:00:00'), Timestamp('2020-02-23 00:00:00')]


In [9]:
import importlib.metadata as importlib_metadata

print(
    "tzdata version: "
    f"{importlib_metadata.version('tzdata')}"
)

tzdata version: 2025.2
